# Task 1: Baseline CNN & End-to-End Pipeline
____________________________________________________________________________________________________

In this notebook we build the full pipeline on **CIFAR-10** with a CNN trained **from scratch** (no pre-trained weights).

We will do the following steps in order:

1. Settings and device setup
2. Load and normalize the CIFAR-10 train / validation / test sets
3. Define the baseline Convolutional Neural Network
4. Define a loss function and optimizer
5. Train the network (log train/val loss, accuracy and per-class accuracy for each epoch)
6. Plot the learning curves
7. Test the best checkpoint on the unseen test data (accuracy, per-class accuracy, normalized confusion matrix)

**Note:** set `TRAIN = False` to skip the training and load the saved checkpoint and history instead.

## 0. Google Colab setup

Only runs on Google Colab (skipped automatically when running locally).

**Before the first run**, the repository must be cloned into Google Drive (one time only, see the README):
```python
!git clone -b course-project https://github.com/MohammedZaiter/ai.git /content/drive/MyDrive/ai
```

This cell then:
1. mounts Google Drive, so checkpoints and results are saved in `My Drive/ai` and are not lost when the session ends
2. moves into the `Notebooks/` folder so that `import utils` and the relative paths work

On Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/ai"  # folder of the project on Google Drive
    os.chdir(PROJECT_DIR + "/Notebooks")
    print("Working directory:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload: changes in utils.py are used without restarting the kernel

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

import utils
from utils import classes

print(torch.__version__)

## 1. Settings and device setup

In [ ]:
TRAIN = True  # False -> load the saved checkpoint + history instead of training

num_epochs = 20
batch_size = 128
learning_rate = 0.001

CHECKPOINT_PATH = f"{utils.CHECKPOINT_DIR}/task1_baseline.pth"
HISTORY_PATH = f"{utils.RESULTS_DIR}/task1_baseline.json"

utils.set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Loading and normalizing CIFAR-10

CIFAR-10: 60,000 RGB images of size 3x32x32 in 10 classes.

| split | images | from |
|---|---|---|
| train | 45,000 | official train set |
| validation | 5,000 | official train set (fixed random split, seed 42) |
| test | 10,000 | official test set, **only used at the end** |

The validation set is used to follow the training and to choose the best epoch. The test set stays unseen until the final evaluation.

For the baseline there is **no data augmentation**, only:
- `ToTensor()`: pixel values from [0, 255] to [0, 1]
- `Normalize(mean, std)` with the CIFAR-10 statistics (mean = [0.4914, 0.4822, 0.4465], std = [0.2470, 0.2435, 0.2616]) so each channel has mean 0 and std 1

\begin{align}output[channel]=\frac{(input[channel] - mean[channel])}{std[channel]}\end{align}

In [ ]:
train_loader, val_loader, test_loader = utils.get_dataloaders(batch_size=batch_size, augmentation="none")

print("train images:", len(train_loader.dataset))
print("val images:", len(val_loader.dataset))
print("test images:", len(test_loader.dataset))
print("Classes:", classes)

**Let us show some of the training images.**

In [ ]:
data_iter = iter(train_loader)
images, labels = next(data_iter)

print('batch size:', images.size(0))
print('color channels :', images.size(1))
print('Image size:' + str(images.size(2)) + 'x' + str(images.size(3)))

# show the first 16 images of the batch
utils.imshow(torchvision.utils.make_grid(images[:16], nrow=16))
print(' '.join('%5s' % classes[labels[j]] for j in range(16)))

## 3. Define the baseline Convolutional Neural Network

Output size of a conv layer (Lab 9):

\begin{align}h_{out}=\frac{h_{in} + 2 \times p - k}{s} + 1 \end{align}

and a `MaxPool2d(2, 2)` divides the height and width by 2.

| layer | details | output shape |
|---|---|---|
| input | RGB image | 3 x 32 x 32 |
| conv1 + ReLU | 32 filters, 3x3, padding 1 | 32 x 32 x 32 |
| max pool | 2x2 | 32 x 16 x 16 |
| conv2 + ReLU | 64 filters, 3x3, padding 1 | 64 x 16 x 16 |
| max pool | 2x2 | 64 x 8 x 8 |
| conv3 + ReLU | 128 filters, 3x3, padding 1 | 128 x 8 x 8 |
| max pool | 2x2 | 128 x 4 x 4 |
| flatten | 128 x 4 x 4 | 2048 |
| fc1 + ReLU | | 256 |
| fc2 | | 10 (one score per class) |

No BatchNorm, no dropout and no augmentation on purpose: this is the reference model that Task 2 tries to improve.

In [ ]:
net = utils.Net().to(device)
print(net)

print("\nTotal parameters:", utils.count_parameters(net))
print("Receptive field:", utils.receptive_field(net), "pixels")

# check the output shape with a random input
dummy_input = torch.randn(1, 3, 32, 32).to(device)
print("output shape:", net(dummy_input).shape)  # [batch size, 10 classes]

## 4. Define a Loss function and optimizer

- **Loss**: Cross-Entropy (softmax + negative log-likelihood), computed on the raw scores of the network
- **Optimizer**: Adam with learning rate 0.001, no scheduler

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=learning_rate)

## 5. Train the network

For each epoch we record: train loss, train accuracy, validation loss, validation accuracy and per-class validation accuracy.

The model with the best validation accuracy is saved to `Checkpoints/task1_baseline.pth`.

In [ ]:
if TRAIN:
    print("Start training")
    history = utils.train_model(net, train_loader, val_loader, criterion, optimizer, num_epochs, device,
                                CHECKPOINT_PATH, checkpoint_info={"model_name": "Net"})
    utils.save_history(history, HISTORY_PATH)
else:
    history = utils.load_history(HISTORY_PATH)
    print("Loaded history from", HISTORY_PATH)

## 6. Learning curves

In [ ]:
utils.plot_history(history, title="Baseline CNN", save_name="task1_curves.png")

best = history["best_epoch"] - 1
print("Best epoch:", best + 1)
print(f"Train acc: {history['train_acc'][best]:.4f} | Val acc: {history['val_acc'][best]:.4f}")
print(f"Generalization gap (train acc - val acc): {history['train_acc'][best] - history['val_acc'][best]:.4f}")

In [ ]:
# per-class validation accuracy recorded at each epoch
utils.plot_class_acc_history(history, save_name="task1_class_acc_history.png")

## 7. Test the network on the unseen test data

We load back the **best checkpoint** (best validation accuracy), not the model of the last epoch.

In [ ]:
net, checkpoint = utils.load_checkpoint(CHECKPOINT_PATH, device)
print(f"Loaded best model from epoch {checkpoint['epoch']} (val acc {checkpoint['val_acc']:.4f})")

test_loss, test_acc, test_class_acc, test_preds, test_labels = utils.evaluate(net, test_loader, criterion, device)
macro_f1 = f1_score(test_labels, test_preds, average="macro")

print('Accuracy of the network on the 10000 test images: %.2f %%' % (100 * test_acc))
print(f"Test loss: {test_loss:.4f} | Macro F1: {macro_f1:.4f}")

**What are the classes that performed well, and the classes that did not perform well?**

In [ ]:
for i in range(10):
    print('Accuracy of %5s : %.1f %%' % (classes[i], 100 * test_class_acc[i]))

utils.plot_class_accuracy(test_class_acc, title="Baseline CNN - per-class test accuracy",
                          save_name="task1_class_acc_test.png")

**Normalized confusion matrix:** each row is divided by the number of images of that true class, so each row sums to 1 and the diagonal is the per-class accuracy. Values outside the diagonal show which classes are confused with each other.

In [ ]:
utils.plot_confusion_matrix(test_labels, test_preds, title="Baseline CNN - normalized confusion matrix (test)",
                            save_name="task1_confusion_matrix.png")

**Predictions on a test mini-batch**

In [ ]:
images, labels = next(iter(test_loader))
images, labels = images[:16], labels[:16]

with torch.no_grad():
    outputs = net(images.to(device))
    _, predicted = torch.max(outputs, 1)

print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(16)))
print('Predicted:   ', ' '.join('%5s' % classes[predicted[j]] for j in range(16)))
utils.show_predictions(images, labels, predicted.cpu())

## Summary

In [ ]:
summary = {
    "model": "Baseline CNN",
    "parameters": utils.count_parameters(net),
    "latency (ms/image)": round(utils.measure_latency(net, device), 3),
    "peak val acc": round(max(history["val_acc"]), 4),
    "generalization gap": round(history["train_acc"][best] - history["val_acc"][best], 4),
    "test acc": round(test_acc, 4),
    "test macro F1": round(macro_f1, 4),
}
pd.DataFrame([summary])

## Observations

*(to fill after training)*

- **Overfitting:** from which epoch does the validation loss increase while the train loss keeps decreasing?
- **Difficult classes:** which classes are confused the most (e.g. cat / dog, car / truck) and why?
- **Generalization gap:** how big is it? Which techniques should reduce it (BatchNorm, dropout, augmentation -> Task 2)?